# Create Hertz Foundation Awards (FELLOWSHIP PATTERN, FacetWP method-3 via direct POST)

Ingest of the Fannie and John Hertz Foundation's public Fellows Directory at `www.hertzfoundation.org/hertz-community/fellows-directory/`. **1,344 fellow cards** spanning fellowship years **1960-2026** (verified 2026-05-21), extracted via direct POST to the Hertz WordPress FacetWP REST refresh endpoint (`/wp-json/facetwp/v1/refresh`). Single unfiltered query — pagination at 12/page across 112 pages — no facet slicing required since the unfiltered total fits in one query. Method #3 on the runbook ladder (search/index APIs).

**Awarding body:** Fannie and John Hertz Foundation — `F4320308782` (US, DOI `10.13039/100005883`, no ROR)

**Why FacetWP and not the WP REST API:** the Hertz site exposes `/wp/v2/people` with 1,502 records but the `fellowships` and `awards` taxonomies are EMPTY across all sampled records (verified: `/wp/v2/fellowships` returns `[]` zero terms; sampled 50 oldest + 10 newest people, all have `fellowships: []`). The fellowship year is only published via the FacetWP directory card markup `<p class="fellow-lister-school year">N</p>`. The REST API path is a dead end. Documented in the script header so no future contributor re-litigates it.

**Pattern: FELLOWSHIP** (variant of PRIZE PATTERN, runbook §11). Each fellow is a single named recipient — not an organization receiving a grant — so:
- `lead_investigator` is **person-level**: `given_name` + `family_name` parsed from the card's `<h3 class="gb-text fellow-lister-name">` text (runbook §2.4.1 split). `affiliation.name` = the fellow's current position+institution string (`<p class="fellow-lister-school">`).
- `funder_scheme = 'Hertz Fellowship'` (single scheme — the foundation has one fellowship program; the four prize-style sub-awards listed under `/wp/v2/awards` taxonomy have count=0 attached posts and are not ingested here).
- `funding_type = 'fellowship'`.
- `declined` always FALSE — the directory only lists active/alumni fellows; declined applicants are not published.
- One row per (slug, fellowship_year) tuple. Slug is unique per fellow on the Hertz site; `funder_award_id = 'hertz-{slug}-{year}'` guarantees uniqueness (verified `is_unique=True` on the full 1,344-row extraction). Prize-pattern rule (runbook prize table row 6): the script RAISES on slug+year collision rather than warning.

**Amount choice — constant $250,000 USD per fellow:**
The Hertz Fellowship is renewable annually for up to 5 years, with the foundation's own `/hertz-fellowship/fellowship-benefits/` page valuing the total package at **"up to $250,000"** (full PhD tuition equivalent + $38-44K/9-month personal stipend × 5 years + $5K/yr dependent supplement). This is a documented per-fellow ceiling — not an aggregator estimate — so we ship it uniformly across all 1,344 rows. `currency = 'USD'`. The fellowship-benefits source URL is documented in the script header. **§6.7 amount-coverage NOT waived** — expect 100% amount populated.

**Start/end dates:** `start_date = {year}-01-01`, `end_date = {year+4}-12-31` — a 5-year renewable window matching the foundation's benefits page. Date precision is year-only because the directory exposes only the cohort year, not the academic-year start.

**Source authority** — `www.hertzfoundation.org` is the awarding body's own site (no auth, no aggregator). FacetWP REST endpoint replayed via plain HTTP after one Playwright-free XHR shape inspection. Method #3 on the runbook ladder.

**Prerequisites**: Run `scripts/local/hertz_to_s3.py` first to POST through 112 pages (~80 sec wall-clock at 0.6s throttle), write parquet, upload to S3.

**Data source**: `https://www.hertzfoundation.org/hertz-community/fellows-directory/` (HTTP POST to `https://www.hertzfoundation.org/wp-json/facetwp/v1/refresh` with FacetWP `facetwp_refresh` JSON payload, `template=fellows_loop`)

**S3 location**: `s3a://openalex-ingest/awards/hertz/hertz_fellows.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hertz_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hertz/hertz_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.hertz_raw;

## Step 1.5: Inspect raw + money/currency scan + coverage acknowledgment

All source columns from the FacetWP fellow-card extraction. **Verified per-row coverage on the full extraction** (1,344 cards across 112 pages, validated 2026-05-21):
- 100% `funder_award_id` (slug+year unique by design)
- 100% `full_name`, `fellowship_year`
- 99.7% `current_position` (1,340/1,344 — 4 missing position; we keep them with NULL affiliation.name)
- 99.6% `expertise` (1,338/1,344 — 6 missing field-of-study; expertise is concatenated into `description`)
- 100% `amount` = constant $250,000 USD (foundation ceiling, see header)
- 67 distinct fellowship years (1960-2026) — Hertz started 1960

Source columns: `funder_award_id`, `slug`, `full_name`, `given_name`, `family_name`, `fellowship_year`, `current_position`, `expertise`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `profile_url`, `thumbnail_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.hertz_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hertz_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — amount is a constant $250K ceiling per the
-- foundation's fellowship-benefits page. Verify the column shipped
-- correctly (no NULL contamination from missing-position rows).
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.hertz_raw;

In [ ]:
%sql
-- Year coverage sanity — fellowship_year should range 1960-2026 with
-- ~20 fellows/year average; recent cohorts higher (~15-20), older
-- cohorts smaller (early 60s = 1-12 fellows).
SELECT MIN(fellowship_year) AS min_yr, MAX(fellowship_year) AS max_yr,
       COUNT(DISTINCT fellowship_year) AS distinct_years,
       COUNT(*) AS total_fellows
FROM openalex.awards.hertz_raw;

## Step 1.6: Fail-fast — verify Hertz funder row exists

In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP — the funder is missing
-- from openalex.common.funder. Flag back to the user; do not proceed.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320308782;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hertz_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320308782  -- Hertz Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    'Hertz Fellowship' AS funder_scheme,
    'hertz_facetwp' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator is PERSON-LEVEL: the Hertz fellow themselves.
    -- given_name + family_name parsed in the script (runbook §2.4.1).
    -- affiliation.name = the current_position string from the card
    -- (e.g. 'PhD Student, Massachusetts Institute of Technology'); this
    -- is the fellow's CURRENT position as listed on the directory, NOT
    -- the institution where they HELD the fellowship — Hertz fellows
    -- often switch institutions mid-fellowship. Country = 'US' since
    -- the Hertz Fellowship requires US citizenship/permanent residency
    -- (runbook §2.3.1 — use known invariant when present).
    CASE
        WHEN n.full_name IS NULL OR n.full_name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.current_position AS name,
                CAST('US' AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.profile_url AS landing_page_url,  -- Hertz hosts a per-fellow /people/{slug}/ page
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.hertz_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.full_name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 90

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hertz_facetwp' AND priority = 90;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    90 AS priority  -- Hertz priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.hertz_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (every Hertz fellow ships with the documented $250K ceiling). Verified expectations from the 2026-05-21 extraction:
- Row count: **1,344** (full directory)
- 100% `pct_amount` (constant $250,000 per fellow)
- `currencies = [USD]`
- 1 distinct `funder_scheme` ('Hertz Fellowship'), 1 distinct `funding_type` ('fellowship')
- 67 distinct fellowship years (1960-2026)
- 100% `lead_investigator.given_name` + `family_name` populated (FELLOWSHIP pattern — person-level PI)
- `declined = FALSE` for all rows

In [ ]:
%sql
SELECT COUNT(*) AS total_hertz_award_rows FROM openalex.awards.hertz_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.hertz_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.hertz_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.hertz_awards;

In [ ]:
%sql
-- Year distribution — useful for verifying 1960-2026 span and roughly
-- ~20 fellows/year average; early years are sparse (1-12 in 1960-1964)
-- and recent years are denser (~15-19 in 2020-2026).
SELECT start_year,
       COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 1) AS total_usd_millions
FROM openalex.awards.hertz_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year ASC
LIMIT 100;

In [ ]:
%sql
-- Sample notable recent fellows + their parsed PI fields
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS affiliation,
       amount, start_year
FROM openalex.awards.hertz_awards
WHERE start_year >= 2024
ORDER BY start_year DESC, family
LIMIT 12;

In [ ]:
%sql
-- Sample historic fellows (1960s-70s) — sanity-check that we parse
-- older cohort names correctly (common Anglo names, no degrees suffixed).
SELECT id,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       start_year
FROM openalex.awards.hertz_awards
WHERE start_year < 1970
ORDER BY start_year ASC, family
LIMIT 12;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.hertz_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined boolean must be FALSE everywhere — Hertz directory doesn't
-- publish declined applicants. (Schema parity with PRIZE PATTERN.)
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.hertz_awards
GROUP BY declined;